# Notebook 1: Apache Spark Foundations & Distributed Architecture (Student Lab)
### Hands-on Workshop: Apache Spark Foundation & Ingestion Framework (Day 1 Morning)
### Related Presentation Slides: Slides 4 - 13 (Modules 1, 2, and 3)

---

## Learning Objectives:
1. Understand Distributed Computing topology: Driver vs. Executors (Slides 4 - 5).
2. Master Partitions as units of parallelism (Slide 6).
3. Demystify Lazy Evaluation: Transformations vs. Actions and inspect Catalyst plans with `.explain()` (Slide 11).
4. Learn Python Essentials for SQL Engineers & Config-Driven Design (Slides 7 - 9).
5. Initialize SparkSession and build the Generic Data Reader (Slides 10 & 12).


---
## Step 1: Environment Setup in Google Colab
Run the cell below to install PySpark and verify the runtime environment.


In [ ]:
# Install PySpark in Colab environment (takes ~15 seconds)
!pip install -q pyspark

import pyspark
print(f"PySpark version: {pyspark.__version__} installed successfully.")


---
## Step 2: Initialize SparkSession (Related: Slide 5 & Slide 10)

In Apache Spark, the `SparkSession` is the single unified entry point to all Spark functionality (DataFrame API, Spark SQL, catalog management).
* Driver Node: Coordinates the application, builds execution plans, and schedules tasks (Slide 5).
* Executor Nodes (Workers): In Google Colab, we run Spark in `local[*]` mode, where `*` means utilizing all available CPU cores as worker threads.


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("01_Spark_Foundations") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print(f"Spark Master: {spark.sparkContext.master}")
print(f"Spark Application Name: {spark.sparkContext.appName}")
print(f"Spark Version: {spark.version}")


---
## Step 3: Core Concepts — Driver vs. Executors & Partitions (Related: Slide 6)

| Component | Responsibility | Analogous Real-World Role |
| :--- | :--- | :--- |
| Driver | Analyzes code, builds DAG, delegates tasks | Construction Site Manager (creates blueprints) |
| Executors (Workers) | Process assigned data chunks in RAM | Construction Workers (do physical heavy lifting) |
| Partition | A chunk of rows residing on one worker node | A pallet of bricks assigned to one worker |

> Golden Rule of Partitions (Slide 6):
> 1 Partition = 1 CPU task thread on an executor.
> Target partition size in memory/cloud storage: 128 MB – 256 MB.


In [ ]:
# Inspect how Spark partitions data
numbers_rdd = spark.sparkContext.parallelize(range(1, 10001), numSlices=4)
print(f"Total partitions: {numbers_rdd.getNumPartitions()}")

# Inspect elements per partition
partition_counts = numbers_rdd.glom().map(len).collect()
print(f"Elements in each partition: {partition_counts}")


---
## Step 4: Lazy Evaluation — Transformations vs. Actions (Related: Slide 11)

Why is Spark so fast compared to naive Python scripts? Lazy Evaluation:

* Transformations (`select`, `filter`, `withColumn`, `groupBy`, `join`):
  * Spark does not execute anything immediately.
  * Instead, it merely records the recipe into a DAG (Directed Acyclic Graph) logical plan.
* Actions (`count`, `show`, `collect`, `write`, `take`):
  * Triggers the Catalyst Optimizer.
  * Spark compiles the DAG into optimized physical bytecode and runs it on the worker nodes.


In [ ]:
# Step A: Define a transformation (Execution completes in 0.00s because NO DATA IS TOUCHED yet)
df_numbers = spark.range(1, 10_000_000, numPartitions=4)
df_filtered = df_numbers.filter("id % 2 == 0").selectExpr("id", "id * 10 as score")

print("Transformation recorded in DAG. No computation occurred yet.")


In [ ]:
# Step B: Inspect the Catalyst execution plan BEFORE execution
df_filtered.explain(True)


In [ ]:
# Step C: Trigger an ACTION
# Now the workers actually compute the result.
total_count = df_filtered.count()
print(f"Action completed. Total filtered records: {total_count}")


---
## Step 5: Python Essentials for SQL/DWH Engineers & Config-Driven Design (Related: Slide 7 & Slide 9)

In production ingestion frameworks:
* Never hardcode table names, file paths, or column mappings inside Python code.
* Use a Configuration File (`config.json`) mapped to Python structures (Slide 9).


In [ ]:
import json
from dataclasses import dataclass
from typing import List, Dict, Any

# Example JSON configuration for an ingestion pipeline
raw_config_json = '''
{
  "pipeline_name": "bundesliga_match_stats",
  "source_format": "json",
  "source_path": "data/raw/bundesliga_events.json",
  "target_path": "data/processed/player_kpi_parquet",
  "target_partitions": 2,
  "required_columns": ["event_id", "player_name", "team_name", "minute", "event_type"]
}
'''

@dataclass
class PipelineConfig:
    pipeline_name: str
    source_format: str
    source_path: str
    target_path: str
    target_partitions: int
    required_columns: List[str]

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> "PipelineConfig":
        return cls(
            pipeline_name=data["pipeline_name"],
            source_format=data["source_format"],
            source_path=data["source_path"],
            target_path=data["target_path"],
            target_partitions=data["target_partitions"],
            required_columns=data["required_columns"]
        )

cfg = PipelineConfig.from_dict(json.loads(raw_config_json))
print(f"Config loaded: Pipeline '{cfg.pipeline_name}', Target: '{cfg.target_path}'")


---
## Step 6: Ingest Real StatsBomb Open Data (Bundesliga 2023/2024 & Team Financials)

In production data platforms, we ingest real event telemetry from external sources, APIs, and data providers.
Here we ingest authentic match event streams directly from the **StatsBomb Open Data Repository** (Bayer Leverkusen's undefeated 2023/2024 Bundesliga championship season, featuring clashes against Bayern Munich, Borussia Dortmund, VfB Stuttgart, and more).

We also download a single-match raw multi-line JSON file to demonstrate multi-line vs. line-delimited JSON ingestion in Apache Spark.


In [ ]:
import os, json, csv, urllib.request

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# 1. Ingest Authentic StatsBomb Open Data (Matches from 2023/2024 Bundesliga)
# Repository: https://github.com/statsbomb/open-data
matches = [
    (3895074, "Bayern Munich vs Bayer Leverkusen"),
    (3895232, "Bayer Leverkusen vs Bayern Munich"),
    (3895158, "Bayer Leverkusen vs Borussia Dortmund"),
    (3895320, "Bayer Leverkusen vs VfB Stuttgart"),
    (3895292, "Union Berlin vs Bayer Leverkusen"),
    (3895302, "Bayer Leverkusen vs Werder Bremen"),
    (3895333, "Eintracht Frankfurt vs Bayer Leverkusen"),
    (3895348, "Bayer Leverkusen vs Augsburg")
]

print("Downloading authentic StatsBomb Bundesliga match events from GitHub...")
all_events = []
single_match_raw = None

for mid, title in matches:
    url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{mid}.json"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            content = resp.read()
            evs = json.loads(content.decode("utf-8"))
            if mid == 3895158:
                single_match_raw = content.decode("utf-8")
            for e in evs:
                e["match_id"] = mid
                e["match_title"] = title
                e["event_id"] = e.get("id")
                all_events.append(e)
            print(f"  + Ingested {title} ({mid}): {len(evs)} events")
    except Exception as err:
        print(f"  ! Warning: {title} ({err})")

# Write line-delimited JSON (optimal for distributed Spark reading)
events_path = "data/raw/bundesliga_events.json"
if all_events:
    with open(events_path, "w", encoding="utf-8") as f:
        for ev in all_events:
            f.write(json.dumps(ev) + "\n")
    print(f"\nTotal {len(all_events)} real StatsBomb events written to '{events_path}'.")

# Also save raw multi-line JSON match for multi-line JSON demonstrations (Slide 12)
if single_match_raw:
    with open("data/raw/statsbomb_match_3895158.json", "w", encoding="utf-8") as f:
        f.write(single_match_raw)

# 2. Ingest Complete Bundesliga Financials & Stadium Lookup Table
financials = [
    {"team_name": "Bayern Munich", "budget_eur": "850000000", "budget_eur_millions": "850.0", "season": "2023/2024", "stadium_capacity": "75024"},
    {"team_name": "Borussia Dortmund", "budget_eur": "520000000", "budget_eur_millions": "520.0", "season": "2023/2024", "stadium_capacity": "81365"},
    {"team_name": "Bayer Leverkusen", "budget_eur": "380000000", "budget_eur_millions": "380.0", "season": "2023/2024", "stadium_capacity": "30210"},
    {"team_name": "RB Leipzig", "budget_eur": "390000000", "budget_eur_millions": "390.0", "season": "2023/2024", "stadium_capacity": "47069"},
    {"team_name": "VfB Stuttgart", "budget_eur": "210000000", "budget_eur_millions": "210.0", "season": "2023/2024", "stadium_capacity": "60449"},
    {"team_name": "Eintracht Frankfurt", "budget_eur": "240000000", "budget_eur_millions": "240.0", "season": "2023/2024", "stadium_capacity": "58000"},
    {"team_name": "Union Berlin", "budget_eur": "160000000", "budget_eur_millions": "160.0", "season": "2023/2024", "stadium_capacity": "22012"},
    {"team_name": "Werder Bremen", "budget_eur": "140000000", "budget_eur_millions": "140.0", "season": "2023/2024", "stadium_capacity": "42100"},
    {"team_name": "Augsburg", "budget_eur": "125000000", "budget_eur_millions": "125.0", "season": "2023/2024", "stadium_capacity": "30660"}
]
with open("data/raw/team_financials.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["team_name", "budget_eur", "budget_eur_millions", "season", "stadium_capacity"])
    writer.writeheader()
    writer.writerows(financials)

print("Saved complete Bundesliga team financials lookup to 'data/raw/team_financials.csv'.")


### Note on StatsBomb Python Wrapper (`statsbombpy`)
In addition to direct JSON ingestion, StatsBomb provides a Python package `statsbombpy`:
```python
# !pip install -q statsbombpy
# from statsbombpy import sb
# matches = sb.matches(competition_id=9, season_id=281)
# events_df = sb.events(match_id=3895158)
```
In our workshop, we ingest the **raw nested JSON event streams** to master PySpark struct navigation, nested schemas, and Catalyst optimization on real cloud payloads.


---
## Step 7: The Generic Reader Pattern (Related: Slide 12)

Build the `DataReader` class using the Strategy Pattern to dynamically read either JSON or CSV based on configuration.


In [ ]:
from pyspark.sql import DataFrame

class DataReader:
    def __init__(self, spark_session: SparkSession):
        self.spark = spark_session

    def read(self, file_format: str, path: str) -> DataFrame:
        fmt = file_format.lower().strip()
        if fmt == "json":
            return self.spark.read.json(path)
        elif fmt == "csv":
            return self.spark.read.option("header", "true").option("inferSchema", "true").csv(path)
        elif fmt == "parquet":
            return self.spark.read.parquet(path)
        else:
            raise ValueError(f"Unsupported format: {file_format}")

reader = DataReader(spark)

# Read JSON events
df_events = reader.read("json", "data/raw/bundesliga_events.json")
print(f"Total events loaded: {df_events.count()}")
df_events.printSchema()

# Read CSV financials
df_financials = reader.read("csv", "data/raw/team_financials.csv")
df_financials.show(truncate=False)


---
## Hands-on Lab Exercise 1 (Corresponding to Lab 1 in Trainer Handbook)

### Task:
1. Filter the events to find only events of type 'Shot' that occurred in the second half (minute > 45).
2. Select `event_id`, `minute`, `player.name`, `type.name` and print the total count.


In [ ]:
# TODO: Write your solution below:
# Hint: use .filter() with expression "type.name == 'Shot' AND minute > 45"
# Assign your result to df_shots_2nd_half

# df_shots_2nd_half = ...

# Verification:
# print(f"Second half shots count: {df_shots_2nd_half.count()}")
# df_shots_2nd_half.select("event_id", "minute", "player.name", "type.name").show(5)
